# AuK Audiobooks on a free GPU

Turns your PDFs into audiobooks in minutes instead of hours, using a free Colab GPU, and keeps
your library in your Google Drive so it is still there next time.

**Before the first run:** put the project in your Drive as `MyDrive/AuKAudiobooks/auk-audiobook`
(upload `auk-audiobook.zip`, made by `deploy/make_colab_zip.sh`, into `MyDrive/AuKAudiobooks/` and
step 1 unpacks it), or set `GIT_URL` below to your own copy of the repository.

**Every run:**
1. `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
2. Step 3 prints a link. Open it on your iPhone in Safari and tap `Share > Add to Home Screen`.
3. Add a book, pick a voice, listen. A full book takes roughly 10-20 minutes.
4. Tap **Download for offline** on a finished book so you can keep listening after you close Colab.

Leave this tab open while you use the app: when Colab disconnects, the server stops (your books
and stats stay in Drive). Free Colab GPUs are not guaranteed and sessions end after a few hours.

In [ ]:
#@title 1. Set up: Drive, project and packages (about 3 minutes)
GIT_URL = ""  #@param {type:"string"}
import os, pathlib, shutil, subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none found. Use Runtime > Change runtime type > T4 GPU, then run again.")

from google.colab import drive
drive.mount("/content/drive")
ROOT = pathlib.Path("/content/drive/MyDrive/AuKAudiobooks")
(ROOT / "data").mkdir(parents=True, exist_ok=True)
(ROOT / "models").mkdir(exist_ok=True)
PROJECT = ROOT / "auk-audiobook"

if not (PROJECT / "server" / "audiobook").exists():
    zips = sorted(ROOT.glob("auk-audiobook*.zip"))
    if GIT_URL:
        subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(PROJECT)], check=True)
    elif zips:
        print("unpacking", zips[-1].name)
        shutil.unpack_archive(str(zips[-1]), str(ROOT))
    else:
        raise SystemExit("Put auk-audiobook.zip in Drive > AuKAudiobooks (or set GIT_URL above).")
print("project:", PROJECT)

%pip install -q "pipecat-ai[kokoro]" fastapi "uvicorn[standard]" python-multipart pymupdf numpy imageio-ffmpeg soundfile
%pip uninstall -y -q onnxruntime onnxruntime-gpu
%pip install -q onnxruntime-gpu
import onnxruntime
print("onnxruntime", onnxruntime.__version__, "providers:", onnxruntime.get_available_providers())

In [ ]:
#@title 2. Kokoro voice model (downloaded once into your Drive)
import pathlib, urllib.request

ROOT = pathlib.Path("/content/drive/MyDrive/AuKAudiobooks")
FILES = {
    "kokoro-v1.0.onnx": "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx",
    "voices-v1.0.bin": "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin",
}
for name, url in FILES.items():
    target = ROOT / "models" / name
    if target.exists() and target.stat().st_size > 1_000_000:
        print("have", name, f"{target.stat().st_size/1e6:.0f} MB")
        continue
    print("downloading", name)
    urllib.request.urlretrieve(url, target)
    print("  saved", f"{target.stat().st_size/1e6:.0f} MB")

In [ ]:
#@title 3. Start the app and print the link for your phone (keep this cell running)
import os, pathlib, re, secrets, subprocess, sys, threading, time, urllib.request

ROOT = pathlib.Path("/content/drive/MyDrive/AuKAudiobooks")
PROJECT = ROOT / "auk-audiobook"
code_file = ROOT / "access-code.txt"
access_code = code_file.read_text().strip() if code_file.exists() else secrets.token_urlsafe(9)
code_file.write_text(access_code)

env = dict(
    os.environ,
    AUDIOBOOK_ENGINE="kokoro",
    AUDIOBOOK_DATA=str(ROOT / "data"),
    AUDIOBOOK_TOKEN=access_code,
    KOKORO_MODEL_PATH=str(ROOT / "models" / "kokoro-v1.0.onnx"),
    KOKORO_VOICES_PATH=str(ROOT / "models" / "voices-v1.0.bin"),
)
server = subprocess.Popen([sys.executable, "-m", "audiobook", "--port", "8000"],
                          cwd=str(PROJECT / "server"), env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

cloudflared = pathlib.Path("/content/cloudflared")
if not cloudflared.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)
tunnel = subprocess.Popen([str(cloudflared), "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        public_url = found.group(0)
        break

for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=2)
        break
    except Exception:
        time.sleep(1)

print("\n" + "=" * 64)
print(" Open this on your iPhone in Safari, then Share > Add to Home Screen:")
print(f"   {public_url}/?pair={access_code}")
print(f" Access code: {access_code}   (same every time, only the link changes)")
print("=" * 64 + "\n")

threading.Thread(target=lambda: [print(l, end="") for l in tunnel.stdout], daemon=True).start()
try:
    for line in server.stdout:  # keeps the cell (and the session) alive
        print(line, end="")
except KeyboardInterrupt:
    server.terminate()
    tunnel.terminate()
    print("stopped")

### When you're done

Stop the cell above (or `Runtime > Disconnect`). The server stops; your books, audio and stats stay
in `MyDrive/AuKAudiobooks/data`, so the next session picks up where you left off. The link changes
every session: open the new one on your phone once and the app remembers it.

**To keep listening with Colab off,** open a finished book and tap **Download for offline** (or
**Download MP3**) before you close this tab.